# Crowd Counting with CSRNet and OpenVINO™

[CSRNet](https://arxiv.org/abs/1802.10062) (*Dilated Convolutional Neural Networks
for Understanding the Highly Congested Scenes*, CVPR 2018) estimates the number of
people in an image by regressing a **density map** whose integral equals the crowd
count. It pairs a VGG-16 front-end with a dilated-convolution back-end that keeps a
large receptive field without losing spatial resolution.

This notebook shows how to:

1. Load a pretrained CSRNet model (PyTorch).
2. Convert it to **OpenVINO IR** at **FP32** and **FP16**.
3. Quantize it to **INT8** with **NNCF** post-training quantization.
4. Run inference on **CPU / GPU / NPU** and visualize the predicted density map.
5. Evaluate the result with **count error, PSNR and SSIM** against the ground truth.

For illustration, we use a single test image from **ShanghaiTech Part A** whose **ground-truth count is 141**.

#### Table of contents
- [Installation Instructions](#Installation-Instructions)
- [Prerequisites](#Prerequisites)
- [Imports](#Imports)
- [Download the model](#Download-the-model)
- [Preprocessing and ground-truth density map](#Preprocessing-and-ground-truth-density-map)
- [Convert to OpenVINO IR (FP32 & FP16)](#Convert-to-OpenVINO-IR-FP32-and-FP16)
- [Quantize to INT8 with NNCF](#Quantize-to-INT8-with-NNCF)
- [Select inference device](#Select-inference-device)
- [Run inference](#Run-inference)
- [Visualize density maps](#Visualize-density-maps)
- [Quantitative evaluation: count, PSNR, SSIM](#Quantitative-evaluation)
- [Latency benchmark](#Latency-benchmark)
- [Conclusion](#Conclusion)


## Installation Instructions
[back to top ⬆️](#Table-of-contents)
 
This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start. For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents)

All dependencies are pinned in [`requirements.txt`](requirements.txt).

The cell below creates an isolated Conda environment named `csrnet-env`, installs the
requirements into it, and registers it as a Jupyter kernel. The setup is guarded by a
`SETUP` flag (default `False`) so that **Run All** skips it; set `SETUP = True` only
when you need to (re)create the environment. When it finishes, select
**"Python (csrnet-env)"** as the notebook kernel (*Kernel ▸ Change Kernel*) and re-run
from the top. If you already have a suitable environment, you can instead just run
`%pip install -q -r requirements.txt` in the active kernel.

In [ ]:
# Set SETUP = True only when you need to (re)create the environment.
# Keep it False for normal "Run All" sessions so setup is skipped.
SETUP = False

if SETUP:
    # Create an isolated, CPU-only Conda environment and install the pinned deps.
    # PyTorch is pulled from the CPU wheel index (see requirements.txt).
    # `conda run` installs into the env without needing an interactive `activate`.
    get_ipython().run_line_magic("conda", "create -y -n csrnet-env python=3.11")
    !conda run -n csrnet-env python -m pip install -r requirements.txt
    !conda run -n csrnet-env python -m ipykernel install --user --name csrnet-env --display-name "Python (csrnet-env)"

# Already inside your target environment? Install in place instead of the above:
# %pip install -q -r requirements.txt

## Imports
[back to top ⬆️](#Table-of-contents)

In [ ]:
import warnings
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import scipy.spatial
from scipy.io import loadmat
from scipy.ndimage import gaussian_filter
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models

import openvino as ov
import nncf
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

warnings.filterwarnings("ignore")
nncf.set_log_level(40)  # quieter NNCF logs
%matplotlib inline

core = ov.Core()
DATA_DIR = Path("data")
MODEL_DIR = Path("model")
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
print("OpenVINO:", ov.__version__)

## Download the model
[back to top ⬆️](#Table-of-contents)

This example needs **three files**:

| File | What it is | Source |
|---|---|---|
| `IMG_114.jpg` | ShanghaiTech Part A test image (GT count = 141) | ShanghaiTech Part A dataset |
| `GT_IMG_114.mat` | head-point annotations for the ground-truth density map | ShanghaiTech Part A dataset |
| `PartAmodel_best.pth.tar` | pretrained CSRNet Part A weights | [Google Drive](https://drive.google.com/file/d/1Z-atzS5Y2pOd-nEWqZRVBDMYJDreGWHH/view) — the CSRNet authors' [reference repo](https://github.com/leeyeehoo/CSRNet-pytorch) |

The test image and its ground-truth file is provided at `data/`. The weights file is fetched from the authors' Google Drive with `gdown` and saved at `data/`. The download is verified against a known **SHA-256** and is skipped when a valid copy already exists.

The ShanghaiTech dataset can be found [here.](https://drive.google.com/file/d/16dhJn7k4FWVwByRsQAEpl9lwjuV03jVI/view)

In [ ]:
import hashlib
import requests
import gdown

# The test image and the corresponding ground-truth is taken from the ShanghaiTech Part A dataset and stored in data/.
# Pretrained CSRNet ShanghaiTech Part A weights published by the CSRNet authors 
# can be downloaded from their GitHub repository (https://github.com/leeyeehoo/CSRNet-pytorch).
# CSRNet weights are pulled from the authors' Google Drive into data/.
# Direct link: https://drive.google.com/file/d/1Z-atzS5Y2pOd-nEWqZRVBDMYJDreGWHH/view
WEIGHTS_GDRIVE_ID = "1Z-atzS5Y2pOd-nEWqZRVBDMYJDreGWHH"

# filename -> expected SHA-256 (pins the exact file and guards against corruption)
SHA256 = {
    "PartAmodel_best.pth.tar": "68383c1053be371ad54b0061ab99fed08c2bfff39de73937f2a4388041ab0128",
}
GT_COUNT = 141  # ground-truth count for this scene


def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _verify(path, name):
    expected = SHA256.get(name)
    if expected and _sha256(path) != expected:
        Path(path).unlink(missing_ok=True)
        raise ValueError(f"checksum mismatch for {name} - download corrupted or wrong source")
    return path


def fetch_weights():
    """Download the CSRNet Part A weights from Google Drive."""
    name = "PartAmodel_best.pth.tar"
    path = DATA_DIR / name
    if path.exists() and _sha256(path) == SHA256[name]:
        print(f"✓ found {path}")
        return path
    print(f"↓ downloading {name} from Google Drive (id={WEIGHTS_GDRIVE_ID})")
    gdown.download(id=WEIGHTS_GDRIVE_ID, output=str(path), quiet=False)
    _verify(path, name)
    print(f"✓ saved {path}")
    return path


IMAGE_FILE = DATA_DIR / "IMG_114.jpg"
ANNOT_FILE = DATA_DIR / "GT_IMG_114.mat"
WEIGHTS_FILE = fetch_weights()

In [ ]:
def make_layers(cfg, in_channels=3, dilation=False):
    rate = 2 if dilation else 1
    layers = []
    for v in cfg:
        if v == "M":
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        else:
            layers += [nn.Conv2d(in_channels, v, kernel_size=3, padding=rate, dilation=rate),
                       nn.ReLU(inplace=True)]
            in_channels = v
    return nn.Sequential(*layers)


class CSRNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.frontend = make_layers([64, 64, "M", 128, 128, "M", 256, 256, 256, "M", 512, 512, 512])
        self.backend = make_layers([512, 512, 512, 256, 128, 64], in_channels=512, dilation=True)
        self.output_layer = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        x = self.output_layer(x)
        return x


def load_pretrained(weights_path):
    model = CSRNet()
    ckpt = torch.load(str(weights_path), map_location="cpu", weights_only=False)
    state = ckpt.get("state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    clean = {k[7:] if k.startswith("module.") else k: v for k, v in state.items()}
    model.load_state_dict(clean, strict=True)
    model.eval()
    return model


torch_model = load_pretrained(WEIGHTS_FILE)
print("CSRNet loaded — parameters:", sum(p.numel() for p in torch_model.parameters()))

## Preprocessing and ground-truth density map
[back to top ⬆️](#Table-of-contents)

The model expects an RGB image normalized with ImageNet statistics. Spatial dims are
cropped to a multiple of 8 (three 2× pooling stages).

The **ground-truth density map** for ShanghaiTech Part A is built from the head-point
annotations using a *geometry-adaptive Gaussian kernel* (σ scaled by the mean distance
to each head's 3 nearest neighbours), exactly as in the CSRNet paper. Its sum equals
the number of annotated heads.

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)


def preprocess(pil_img, factor=8):
    x = np.asarray(pil_img.convert("RGB"), dtype=np.float32) / 255.0  # HWC
    x = np.transpose(x, (2, 0, 1))[None]                              # 1CHW
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    _, _, h, w = x.shape
    x = x[:, :, : h - h % factor, : w - w % factor]                  # crop to /8
    return np.ascontiguousarray(x, dtype=np.float32)


def gt_density_map(shape, points):
    # Geometry-adaptive (k-NN) Gaussian density map - ShanghaiTech Part A.
    h, w = shape
    density = np.zeros((h, w), dtype=np.float32)
    pts = np.asarray(points, dtype=np.float32)
    n = len(pts)
    if n == 0:
        return density
    tree = scipy.spatial.KDTree(pts.copy(), leafsize=2048)
    distances, _ = tree.query(pts, k=min(4, n))
    for i, p in enumerate(pts):
        x = min(w - 1, max(0, int(round(p[0]))))
        y = min(h - 1, max(0, int(round(p[1]))))
        pt = np.zeros((h, w), dtype=np.float32)
        pt[y, x] = 1.0
        if n > 3:
            sigma = (distances[i][1] + distances[i][2] + distances[i][3]) * 0.1
        else:
            sigma = np.average([h, w]) / 4.0
        density += gaussian_filter(pt, sigma, mode="constant")
    return density


# Load image + annotations
pil_image = Image.open(IMAGE_FILE)
image_rgb = np.asarray(pil_image.convert("RGB"))
points = loadmat(ANNOT_FILE)["image_info"][0][0][0][0][0]

input_tensor = preprocess(pil_image)
gt_density = gt_density_map(image_rgb.shape[:2], points)

print(f"image size (HxW): {image_rgb.shape[:2]}")
print(f"model input shape: {tuple(input_tensor.shape)}")
print(f"annotated heads (ground-truth count): {len(points)}")
print(f"ground-truth density map sum: {gt_density.sum():.2f}")

## Convert to OpenVINO IR (FP32 and FP16)
[back to top ⬆️](#Table-of-contents)

`ov.convert_model` traces the PyTorch model. We keep the spatial dimensions **dynamic**
(`[1, 3, ?, ?]`) so any image resolution works. Saving with `compress_to_fp16=True`
stores the weights in half precision (≈2× smaller) without changing the graph.

In [ ]:
IR_FP32 = MODEL_DIR / "csrnet_fp32.xml"
IR_FP16 = MODEL_DIR / "csrnet_fp16.xml"

example = torch.zeros(1, 3, 768, 1024, dtype=torch.float32)
ov_model = ov.convert_model(torch_model, example_input=example, input=[[1, 3, -1, -1]])
ov_model.inputs[0].get_tensor().set_names({"image"})
ov_model.outputs[0].get_tensor().set_names({"density_map"})

ov.save_model(ov_model, IR_FP32, compress_to_fp16=False)
ov.save_model(ov_model, IR_FP16, compress_to_fp16=True)
print("FP32 IR:", round(IR_FP32.with_suffix('.bin').stat().st_size / 1e6, 1), "MB")
print("FP16 IR:", round(IR_FP16.with_suffix('.bin').stat().st_size / 1e6, 1), "MB")

## Quantize to INT8 with NNCF
[back to top ⬆️](#Table-of-contents)

NNCF post-training quantization needs a small **calibration dataset** to record
activation ranges. Production use should calibrate on tens to hundreds of
representative images; since this notebook downloads only a single image, we build a
lightweight calibration set from random crops of that image purely to demonstrate the
INT8 flow.

In [ ]:
def make_calibration_set(pil_img, n=16, crop=(576, 768), seed=0):
    rng = np.random.default_rng(seed)
    arr = np.asarray(pil_img.convert("RGB"))
    H, W = arr.shape[:2]
    ch, cw = min(crop[0], H), min(crop[1], W)
    samples = []
    for _ in range(n):
        top = rng.integers(0, H - ch + 1)
        left = rng.integers(0, W - cw + 1)
        patch = Image.fromarray(arr[top:top + ch, left:left + cw])
        samples.append(preprocess(patch))
    return samples


calib = make_calibration_set(pil_image)
calibration_dataset = nncf.Dataset(calib, lambda x: x)

int8_model = nncf.quantize(
    core.read_model(IR_FP32),
    calibration_dataset,
    preset=nncf.QuantizationPreset.MIXED,
    subset_size=len(calib),
    fast_bias_correction=False,
)
IR_INT8 = MODEL_DIR / "csrnet_int8.xml"
ov.save_model(int8_model, IR_INT8)
print("INT8 IR:", round(IR_INT8.with_suffix('.bin').stat().st_size / 1e6, 1), "MB")

## Select inference device
[back to top ⬆️](#Table-of-contents)

Pick the OpenVINO device. `AUTO` lets the runtime choose the best available target
(GPU/NPU/CPU).

In [ ]:
import ipywidgets as widgets

device = widgets.Dropdown(
    options=core.available_devices + ["AUTO"],
    value="AUTO",
    description="Device:",
    disabled=False,
)
device

## Run inference
[back to top ⬆️](#Table-of-contents)

First run the original **PyTorch** model to obtain a reference density map, then
compile each OpenVINO precision on the selected device. In every case, the count is simply the sum of the density map. The PyTorch result is considered as the baseline and the FP32/FP16/INT8 OpenVINO IRs are compared against that.

In [ ]:
# Reference inference with the original PyTorch model.
def infer_torch(model, tensor):
    with torch.no_grad():
        out = model(torch.from_numpy(tensor))
    return out[0, 0].cpu().numpy()  # HxW density map

# Reference inference with the OpenVINO models.
def infer(ir_path, tensor, device_name, precision="FP32"):
    config = {}
    hint = _PRECISION_HINTS.get(precision)
    if hint is not None:
        config["INFERENCE_PRECISION_HINT"] = hint
    compiled = core.compile_model(core.read_model(ir_path), device_name, config)
    out = compiled(tensor)[compiled.output(0)]
    return out[0, 0]  # HxW density map

# INFERENCE_PRECISION_HINT tells the device which compute precision to use.
# It only accepts floating-point hints (f32 / f16 / bf16); "int8" is NOT a
# valid value, so for the INT8 model we omit the hint and let the device run
# the already-quantized weights at their native precision.
_PRECISION_HINTS = {"FP32": "f32", "FP16": "f16", "INT8": None}

pred_maps = {"PyTorch": infer_torch(torch_model, input_tensor)}
precisions = {"FP32": IR_FP32, "FP16": IR_FP16, "INT8": IR_INT8}
pred_maps.update({name: infer(path, input_tensor, device.value, name) for name, path in precisions.items()})

for name, dmap in pred_maps.items():
    print(f"{name:<8}: predicted count = {dmap.sum():7.2f}   (density map {dmap.shape})")

# PyTorch baseline first, then the OpenVINO precisions.pred_maps = {"PyTorch": infer_torch(torch_model, input_tensor)}

## Visualize density maps
[back to top ⬆️](#Table-of-contents)

For illustration, we have the input image, the ground-truth density map, and the OpenVINO **FP16** density map. Brighter regions correspond to higher person density.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 5))
axes[0].imshow(image_rgb)
axes[0].set_title("Input image")
axes[1].imshow(gt_density, cmap="jet")
axes[1].set_title(f"Ground truth  (count = {gt_density.sum():.0f})")
axes[2].imshow(pred_maps["FP16"], cmap="jet")
axes[2].set_title(f"CSRNet FP16  (count = {pred_maps['FP16'].sum():.0f})")
for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Quantitative evaluation
[back to top ⬆️](#Table-of-contents)

We report, for every precision:

- **GT count** and **estimated count** (sum of the density map) and the absolute error.
- **PSNR** and **SSIM** between the predicted and ground-truth density maps — as used in the CSRNet paper. The prediction (1/8 resolution) is resized to the ground-truth resolution with count preserved, then both maps are normalized to a shared `[0, 1]` scale before computing the metrics.

In [ ]:
def density_quality(pred_map, gt_map):
    gh, gw = gt_map.shape
    pred_r = cv2.resize(pred_map.astype(np.float32), (gw, gh), interpolation=cv2.INTER_CUBIC)
    if pred_r.sum() > 0:                       # preserve the total count after resize
        pred_r *= pred_map.sum() / pred_r.sum()
    scale = max(gt_map.max(), pred_r.max(), 1e-8)
    gt_n = (gt_map / scale).astype(np.float32)
    pred_n = np.clip(pred_r / scale, 0, 1).astype(np.float32)
    psnr = peak_signal_noise_ratio(gt_n, pred_n, data_range=1.0)
    ssim = structural_similarity(gt_n, pred_n, data_range=1.0)
    return psnr, ssim


header = f"{'Precision':<10}{'GT':>6}{'Estimate':>11}{'|Error|':>9}{'PSNR (dB)':>11}{'SSIM':>8}"
print(header)
print("-" * len(header))
for name, dmap in pred_maps.items():
    est = float(dmap.sum())
    psnr, ssim = density_quality(dmap, gt_density)
    print(f"{name:<10}{GT_COUNT:>6}{est:>11.2f}{abs(est - GT_COUNT):>9.2f}{psnr:>11.2f}{ssim:>8.4f}")

## Latency benchmark
[back to top ⬆️](#Table-of-contents)

A quick steady-state latency / throughput measurement per precision on the selected
device (fixed input shape).

In [ ]:
import time


def benchmark(ir_path, tensor, device_name, runs=20, warmup=5):
    compiled = core.compile_model(core.read_model(ir_path), device_name,
                                  {"PERFORMANCE_HINT": "LATENCY"})
    out = compiled.output(0)
    for _ in range(warmup):
        compiled(tensor)
    t0 = time.perf_counter()
    for _ in range(runs):
        compiled(tensor)
    dt = (time.perf_counter() - t0) / runs
    return dt * 1e3, 1.0 / dt


print(f"{'Precision':<10}{'Latency (ms)':>14}{'FPS':>10}")
print("-" * 34)
for name, path in precisions.items():
    ms, fps = benchmark(path, input_tensor, device.value)
    print(f"{name:<10}{ms:>14.2f}{fps:>10.2f}")

## Conclusion
[back to top ⬆️](#Table-of-contents)

We just converted CSRNet to OpenVINO IR (FP32/FP16), quantized it to INT8 with NNCF, ran
it on the selected device, and evaluated the prediction on a scene (test image) from ShanghaiTech Part A 
(GT = 141) using count error, PSNR and SSIM. FP32/FP16/INT8 OpenVINO IRs match the PyTorch model's count closely with INT8 offering upto 2x speedup.

### References
- CSRNet paper: https://arxiv.org/abs/1802.10062
- ShanghaiTech dataset (Zhang *et al.*, CVPR 2016)
- OpenVINO: https://docs.openvino.ai
- NNCF: https://github.com/openvinotoolkit/nncf
